# Tutorial 8B — Pretrained detectors and measured mAP

Run pretrained detection on a public image and compare three models/backbones on the same annotated examples. This implementation uses maintained Torchvision weight APIs instead of the handout’s TensorFlow Hub loading route.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Tutorial-01_Perceptron_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Load data and define the three pretrained models
The baseline is Faster R-CNN with ResNet50; the alternatives use MobileNetV3 and SSD with VGG16. These are COCO-trained models. COCO person is label 1, matching the public pedestrian dataset.

In [ ]:
import torch,gc,time,pandas as pd
from torchvision.models import detection as det
from tutorial_support.detection import *
DEVICE=torch_setup();dataset=PennFudan();train,val,test=loaders(dataset)
specs=[('FasterRCNN ResNet50',lambda:det.fasterrcnn_resnet50_fpn(weights=det.FasterRCNN_ResNet50_FPN_Weights.DEFAULT)),('FasterRCNN MobileNetV3',lambda:det.fasterrcnn_mobilenet_v3_large_320_fpn(weights=det.FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT)),('SSD VGG16',lambda:det.ssd300_vgg16(weights=det.SSD300_VGG16_Weights.DEFAULT))]

## Step 3 — Predict display and evaluate
Input tensors are RGB floats in [0,1]. Torchvision detection models perform their own normalization/resizing. The drawing threshold only hides boxes in the plot; evaluation retains low-score detections to form the precision–recall curve. We filter COCO predictions to person for this one-class dataset.

In [ ]:
rows=[]
for name,constructor in specs:
    model=constructor().to(DEVICE).eval()
    image,target=test.dataset[0]
    with torch.inference_mode():prediction=model([image.to(DEVICE)])[0]
    print(name);show_prediction(image,prediction,.6)
    start=time.perf_counter();metrics=evaluate_detector(model,test,DEVICE,person_only=True)
    rows.append({'model':name,**metrics,'evaluation_seconds':time.perf_counter()-start})
    del model;gc.collect()
display(pd.DataFrame(rows));save_result('08B_pretrained_detection',rows)

## Step 4 — Run on a different image
Replace IMAGE_PATH with your own path if desired. The default sample is public and unlabelled; we can display detections but cannot compute its accuracy or AP without ground truth.

In [ ]:
from PIL import Image
from torchvision.transforms.functional import to_tensor
IMAGE_PATH=sample_image()
image=to_tensor(Image.open(IMAGE_PATH).convert('RGB'))
model=specs[1][1]().to(DEVICE).eval()
with torch.inference_mode():prediction=model([image.to(DEVICE)])[0]
show_prediction(image,prediction,.5)

## Task interpretation
The table compares complete detectors, not just backbone capacity: image resizing, heads and training recipes differ too. A low confidence threshold increases detections and may increase false positives. Pretrained person detection benefits from prior COCO training; this is not evidence that a newly trained custom model has learned equally well.

[Torchvision detection models](https://docs.pytorch.org/vision/stable/models.html#object-detection-instance-segmentation-and-person-keypoint-detection).

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.

## Task answers and revision check

**Three tested designs to compare:** Faster R-CNN with ResNet50, Faster R-CNN with MobileNetV3, and SSD with VGG16. The last changes both detector and backbone; differences cannot be attributed solely to backbone size.

**Precision versus recall:** precision asks how many returned detections are correct; recall asks how many annotated objects were found. A higher confidence threshold can remove false positives but may also miss true objects.

**Why is AP more than one precision value?** It summarizes a confidence-ranked precision–recall curve with one-to-one matching. AP50 uses IoU 0.50; COCO mAP averages AP over IoU thresholds 0.50:0.95. State the protocol with the number.

**Quiz 3 answer:** R-CNN runs a CNN per external proposal. Fast R-CNN shares the image feature map but still uses external proposals. Faster R-CNN learns proposals with an RPN sharing that feature map. RoI pooling makes variable regions a fixed size; it does not generate proposals.

**Result to record after execution:** same-split AP50, AP75, mAP and runtime, plus one boxed prediction per model. None of these values is invented in the current notebook.
